# Tutorial 2 — ASTER Mt. Rainier, orthorectified

> Reading this as a static page? To run the notebook, [open this repo in a GitHub Codespace](https://stereopipeline-quickstart.readthedocs.io/en/latest/start/codespaces.html); ASP and `asp-plot` come pre-installed.

The same ASTER scene and the same window around the summit as [Tutorial 1](01_aster_rainier.ipynb), with one step added before stereo: both images are orthorectified onto a Copernicus reference DEM. Stereo uses Tutorial 1's settings, so the differences between the two DEMs come from orthorectification. The report replays Tutorial 1's figure choices, so the two reports line up panel for panel.

This takes about 10 minutes in a 4-core Codespace. Most of the extra time over Tutorial 1 goes to the reference DEM fetch and orthorectification.

## What we'll do

1. Reuse the scene and cameras from Tutorial 1, or prepare them if you skipped it.
2. Clip a Copernicus DEM tile from AWS Open Data.
3. Orthorectify both views onto it over Tutorial 1's window (ASP calls this "mapprojection") and re-run stereo.
4. Generate a report and compare it with Tutorial 1's.

Scene: ASTER L1A acquired 2017-07-31 over Mt. Rainier, WA.

Reference: [ASP's official ASTER example](https://stereopipeline.readthedocs.io/en/latest/examples/aster.html).


In [ ]:
from pathlib import Path

DATA = "../data/aster_rainier"

!mkdir -p {DATA}

REF_DEM = f"{DATA}/ref/cop30_rainier.tif"

# The area to orthorectify, xmin ymin xmax ymax in UTM zone 10N meters: the
# extent of Tutorial 1's DEM, so both tutorials cover the same ground.
T_PROJWIN = "577305 5172975 615705 5209035"

# ICESat-2 points and figure selections shipped with this tutorial, fetched by
# the download step. Every report replays them with --reuse-selections, so no
# SlideRule request is made. Set FRESH_ICESAT2 = True to request fresh
# ICESat-2 data from SlideRule instead.
FRESH_ICESAT2 = False
SHIPPED_SELECTIONS = f"{DATA}/icesat2_shipped/figure_selections.yml"


## 1. Scene and cameras

The same download and `aster2asp` steps as Tutorial 1. If you ran it, both are skipped.

> In a 4-core Codespace, this step takes a few seconds if you ran Tutorial 1, under a minute if not.

In [ ]:
!bash ../scripts/download_aster.sh {DATA}

if Path(f"{DATA}/out-Band3N.tif").exists():
    print(f"{DATA}/out-Band3N.tif exists — skipping aster2asp. Delete out-Band3*.* to reprocess.")
else:
    !aster2asp {DATA}/dataDir -o {DATA}/out

!ls -1 {DATA}/out-Band3*.tif {DATA}/out-Band3*.xml


## 2. Reference DEM from AWS Copernicus DEM

Orthorectification needs a reference DEM. Copernicus GLO-30 is on AWS Open Data; no API key, no auth.

`scene_bbox` runs ASP's `camera_footprint` on each view and unions the results; `utm_epsg` picks the UTM zone at the bbox center. `fetch_cop_dem.py` then clips, mosaics, and reprojects the matching COP30 tiles.

> In a 4-core Codespace, this step takes under a minute.

In [ ]:
from utils import scene_bbox, utm_epsg

BBOX = scene_bbox(
    (f"{DATA}/out-Band3N.tif", f"{DATA}/out-Band3N.xml"),
    (f"{DATA}/out-Band3B.tif", f"{DATA}/out-Band3B.xml"),
    session="aster",
)
T_SRS = utm_epsg(BBOX)
print(f"Scene bbox: {BBOX}  UTM: {T_SRS}")

Path(f"{DATA}/ref").mkdir(exist_ok=True)

if Path(REF_DEM).exists():
    print(f"{REF_DEM} exists — skipping COP30 fetch. Delete it to redownload.")
else:
    !python ../scripts/fetch_cop_dem.py \
        --bbox {BBOX} \
        --t-srs {T_SRS} \
        --tr 30 \
        --out {REF_DEM}

!ls -lh {REF_DEM}


## 3. Orthorectified pass

Run `mapproject` to orthorectify both images onto the Copernicus DEM at 15 m, and re-run stereo on the orthorectified pair. The matcher's search range is much smaller because the inputs are already aligned to the terrain.

- `--t_projwin {T_PROJWIN}`: orthorectify only the area of Tutorial 1's DEM. The images are then already cropped, so stereo needs no crop window of its own. The area is a little larger than Tutorial 1's window, which sits at an angle to the map grid.

Stereo uses Tutorial 1's settings (`asp_mgm`, `--subpixel-mode 9`, one process with four threads), with one addition:

- `--dem {REF_DEM}`: the DEM the images were orthorectified onto, which stereo needs to triangulate with the original cameras.

> In a 4-core Codespace, this step takes about 8 minutes, most of the tutorial's run time.

In [ ]:
%%time
if Path(f"{DATA}/stereo_ortho/run-DEM.tif").exists():
    print(f"{DATA}/stereo_ortho/run-DEM.tif exists — skipping mapprojected pass. Delete {DATA}/stereo_ortho/ to reprocess.")
else:
    !mapproject --t_srs {T_SRS} --tr 15 --t_projwin {T_PROJWIN} \
        {REF_DEM} \
        {DATA}/out-Band3N.tif {DATA}/out-Band3N.xml \
        {DATA}/out-Band3N_ortho.tif

    !mapproject --t_srs {T_SRS} --tr 15 --t_projwin {T_PROJWIN} \
        {REF_DEM} \
        {DATA}/out-Band3B.tif {DATA}/out-Band3B.xml \
        {DATA}/out-Band3B_ortho.tif

    !parallel_stereo -t aster \
        --stereo-algorithm asp_mgm \
        --subpixel-mode 9 \
        --dem {REF_DEM} \
        --processes 1 --threads-multiprocess 4 \
        {DATA}/out-Band3N_ortho.tif {DATA}/out-Band3B_ortho.tif \
        {DATA}/out-Band3N.xml {DATA}/out-Band3B.xml \
        {DATA}/stereo_ortho/run

    !point2dem --tr 30 -r earth --auto-proj-center --errorimage {DATA}/stereo_ortho/run-PC.tif

!ls -lh {DATA}/stereo_ortho/run-DEM.tif


## 4. Report

The same report as Tutorial 1, for the orthorectified DEM, now with the Copernicus DEM as the reference for difference maps. Tutorial 1 explains `--no-geometry` and `--atl06sr-time-range`.

`--reuse-selections` replays the figure choices of the Tutorial 1 report — the same ICESat-2 sample points, profile track, and hillshade crops. The two reports then line up panel for panel, so the comparison reflects the processing change rather than different figure selections. With `FRESH_ICESAT2 = True`, it uses the sidecar your own Tutorial 1 run left behind, if there is one; otherwise this run makes its own selections.

`--figures-directory` keeps a copy of each report figure as a PNG, and the cell then shows three of them below it.

> In a 4-core Codespace, this step takes under a minute. It runs `pc_align`.


In [ ]:
%%time
from utils import show_report_figures

FIGS = f"{DATA}/stereo_ortho/report_figures"
if Path(f"{DATA}/stereo_ortho/rainier_aster_ortho_report.pdf").exists():
    print(f"{DATA}/stereo_ortho/rainier_aster_ortho_report.pdf exists — skipping asp_report. Delete it to regenerate.")
else:
    prior = f"{DATA}/stereo/rainier_aster_report_figure_selections.yml"
    if not FRESH_ICESAT2:
        prior = SHIPPED_SELECTIONS
    reuse = f"--reuse-selections {prior}" if Path(prior).exists() else ""
    !asp_report \
        --directory {DATA} \
        --stereo-directory stereo_ortho \
        --reference-dem {REF_DEM} \
        --no-geometry \
        --subset-km 5 \
        --atl06sr-time-range 2018-10-14,2020-10-14 \
        {reuse} \
        --figures-directory {FIGS} \
        --report-filename rainier_aster_ortho_report.pdf

show_report_figures(FIGS)


Below the report cell are three of the report's figures. Compare them with the same figures from Tutorial 1: the ICESat-2 profile follows the same track, and the zoomed hillshade details show the same places.

The reports are at:

- Tutorial 1: [data/aster_rainier/stereo/rainier_aster_report.pdf](../data/aster_rainier/stereo/rainier_aster_report.pdf), if you ran it
- This tutorial: [data/aster_rainier/stereo_ortho/rainier_aster_ortho_report.pdf](../data/aster_rainier/stereo_ortho/rainier_aster_ortho_report.pdf)

To save a copy to your computer, right-click the PDF in the file explorer on the left and choose **Download**.

## What's next

→ [Tutorial 3: WorldView-3 UCSD](03_worldview_ucsd.ipynb) adds high-resolution commercial imagery, cropping, and ICESat-2 alignment. [Tutorial 4](04_worldview_ucsd_ba.ipynb) adds bundle adjustment on top of it.

→ For deeper ASTER processing (jitter correction, full bundle adjust), see [`asp-plot`'s ASTER notebooks](https://asp-plot.readthedocs.io/en/latest/examples/notebooks/aster_with_bundle_adjust_and_jitter_correction.html).
